# Route C: redox enzymes in diploid and tetraploid mouse hepatocytes (Lanz et al. 2024, Table S1)

Put Table S1 in *Colab Notebooks/route_c_liver* on your Drive, then **Runtime → Run all**.

In [ ]:
#@title Cell 1 - Settings and connect Google Drive
import os
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/route_c_liver"  #@param {type:"string"}
TABLE_FILE = "NIHMS2150604-supplement-Table_S01.xlsx"  #@param {type:"string"}
DATA_DIR = os.environ.get("ROUTE_C_DATA", DATA_DIR)
if DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
assert os.path.exists(os.path.join(DATA_DIR, TABLE_FILE)), "Put Table S1 in the folder above and check the file name"
print("Ready")

In [ ]:
#@title Cell 2 - Redox enzyme sets: ploidy vs genome concentration vs cell size
import os, numpy as np, pandas as pd
f = os.path.join(DATA_DIR, TABLE_FILE)
d = pd.read_excel(f, sheet_name="Liver Proteomics")
d = d[d["Gene name"].notna()].copy()
d["gene"] = d["Gene name"].astype(str).str.split().str[0]
d = d.drop_duplicates("gene")
L2, L4, S4S2, L4L2 = "large (2N) / small (2N)", "large (4N) / small (4N)", "4N (small) / 2N (small)", "4N (large) / 2N (large)"
d["ploidy_same_size"] = d[S4S2] - d[L2]          # small 4n vs large 2n: same size, 2x DNA per volume
d["ploidy_same_dna_conc"] = d[S4S2]               # small 4n vs small 2n: same DNA per volume, 4n twice as big
d["size_2n"] = d[L2]                               # large vs small 2n: genome diluted
d["size_4n"] = d[L4]
COMP = {"ploidy_same_dna_conc": "4n vs 2n, same DNA per volume (small 4n / small 2n)",
        "ploidy_same_dna_conc_large": "4n vs 2n, large cells (large 4n / large 2n)",
        "ploidy_same_size": "same size, 2x DNA per volume (small 4n / large 2n)",
        "size_2n": "2x bigger, same ploidy 2n (large / small)",
        "size_4n": "2x bigger, same ploidy 4n (large / small)"}
d["ploidy_same_dna_conc_large"] = d[L4L2]
d = d[d.UniqID >= 2].reset_index(drop=True)
SETS = {
 "Glutathione synthesis & recycling": "Gclc Gclm Gss Gsr Gpx1 Gpx4 Glrx Glrx2 Glrx3 Glrx5",
 "Glutathione S-transferases": "Gsta1 Gsta2 Gsta3 Gsta4 Gstm1 Gstm2 Gstm3 Gstm5 Gstm6 Gstm7 Gstp1 Gstp2 Gstk1 Gsto1 Gstt1 Gstt2 Gstt3 Gstz1 Mgst1 Mgst3",
 "Thioredoxin / peroxiredoxin": "Txn1 Txnrd1 Txn2 Txnrd2 Prdx1 Prdx2 Prdx3 Prdx4 Prdx5 Prdx6 Srxn1 Txnl1",
 "Superoxide dismutase & catalase": "Sod1 Sod2 Sod3 Cat",
 "NADPH supply": "G6pdx Pgd Tkt Taldo1 Me1 Me2 Idh1 Idh2 Nnt Mthfd1 Aldh1l1",
 "NRF2 targets (other)": "Nqo1 Hmox1 Ephx1 Cbr1 Cbr3 Akr7a5 Akr1b8 Ftl1 Fth1 Blvrb Ugdh",
 "H2O2 / ROS-producing enzymes": "Cyp2e1 Xdh Maoa Maob Ero1a Ero1lb Acox1 Acox2 Acox3 Hao1 Hao2 Dao Aox1 Nox4",
}
POS = {"CONTROL: histones (track DNA)": d.loc[d.Histone == True, "gene"].tolist(),
       "CONTROL: mitochondrial proteins": d.loc[d.Mitochondria == True, "gene"].tolist()}
present = set(d.gene)
sets = {k: [g for g in v.split() if g in present] for k, v in SETS.items()}
sets.update({k: v for k, v in POS.items()})
rng = np.random.default_rng(1)
# random sets matched on number of peptides (measurement precision)
d["pbin"] = pd.qcut(d.UniqID.rank(method="first"), 10, labels=False)
pool = {b: g.index.to_numpy() for b, g in d.groupby("pbin")}
idx = {g: i for i, g in enumerate(d.gene)}
rows = []
for name, genes in sets.items():
    ii = np.array([idx[g] for g in genes]); bins = d.pbin.to_numpy()[ii]
    null_ix = np.stack([np.array([rng.choice(pool[b]) for b in bins]) for _ in range(5000)])
    for col, lab in COMP.items():
        v = d[col].to_numpy(); obs = np.median(v[ii]); null = np.median(v[null_ix], axis=1)
        p = (1 + np.sum(np.abs(null - np.median(null)) >= abs(obs - np.median(null)))) / (len(null) + 1)
        rows.append({"set": name, "n": len(ii), "comparison": col, "median_log2": obs, "random_median": np.median(null), "p_vs_random": p})
res = pd.DataFrame(rows)
redox = ~res.set.str.startswith("CONTROL")
for col in COMP:
    m = redox & (res.comparison == col); p = res.loc[m, "p_vs_random"].to_numpy(); o = np.argsort(p)
    q = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]; qq = np.empty_like(p); qq[o] = np.minimum(q, 1)
    res.loc[m, "q_fdr"] = qq
pd.set_option("display.width", 250)
print("Proteins analysed:", len(d))
for k, v in sets.items():
    print(f"  {k}: {len(v)} found" + ("" if k.startswith("CONTROL") else f" ({', '.join(v)})"))
piv = res.pivot(index="set", columns="comparison", values="median_log2")[list(COMP)]
print("\nMedian log2 ratio per set:"); print(piv.round(3).to_string())
pv = res.pivot(index="set", columns="comparison", values="p_vs_random")[list(COMP)]
print("\np vs random protein sets:"); print(pv.round(4).to_string())
qv = res.pivot(index="set", columns="comparison", values="q_fdr")[list(COMP)]
print("\nq (FDR across redox sets):"); print(qv.round(3).to_string())
genes = sorted({g for k, v in sets.items() if not k.startswith("CONTROL") for g in v})
gt = d.set_index("gene").loc[genes, ["UniqID"] + list(COMP)]
# per-protein z vs all proteins
for col in COMP:
    gt[col + "_pct"] = d[col].rank(pct=True).to_numpy()[[idx[g] for g in genes]]
gt.to_csv(os.path.join(DATA_DIR, "redox_enzymes_per_protein.csv"))
res.to_csv(os.path.join(DATA_DIR, "redox_sets.csv"), index=False)
print("\nExtreme single enzymes (top/bottom 5% of all proteins) in ploidy_same_size:")
x = gt[(gt.ploidy_same_size_pct > .95) | (gt.ploidy_same_size_pct < .05)]
print(x[["UniqID", "ploidy_same_size", "ploidy_same_dna_conc", "size_2n"]].round(2).sort_values("ploidy_same_size").to_string())
print("\nExtreme in ploidy_same_dna_conc:")
x = gt[(gt.ploidy_same_dna_conc_pct > .95) | (gt.ploidy_same_dna_conc_pct < .05)]
print(x[["UniqID", "ploidy_same_dna_conc", "ploidy_same_dna_conc_large", "ploidy_same_size"]].round(2).sort_values("ploidy_same_dna_conc").to_string())
print("\nAll-protein spread (sd):", d[list(COMP)].std().round(3).to_dict())
print("Correlation size_2n vs ploidy_same_size across all proteins:", round(d.size_2n.corr(d.ploidy_same_size), 3))


In [ ]:
#@title Cell 3 - Chart
import matplotlib.pyplot as plt
order = ["ploidy_same_dna_conc", "ploidy_same_dna_conc_large", "ploidy_same_size", "size_2n", "size_4n"]
labels = ["4n vs 2n\n(small cells)", "4n vs 2n\n(large cells)", "2x DNA per volume\n(same size)", "2x bigger\n(2n)", "2x bigger\n(4n)"]
fig, ax = plt.subplots(figsize=(9, 4.8))
sets_ = [s for s in piv.index]
for j, s in enumerate(sets_):
    for i, c in enumerate(order):
        v = piv.loc[s, c]; q = res[(res.set == s) & (res.comparison == c)].q_fdr.iloc[0]
        p = res[(res.set == s) & (res.comparison == c)].p_vs_random.iloc[0]
        sig = (q < 0.05) if not s.startswith("CONTROL") else (p < 0.01)
        ax.scatter(i, j, s=60 + 400 * min(abs(v), 1.2), c="#b2432f" if v < 0 else "#2f6db2",
                   edgecolors="black" if sig else "none", linewidths=1.5, alpha=0.85)
        ax.text(i + 0.22, j, f"{v:+.2f}", va="center", fontsize=8)
ax.set_xticks(range(len(order))); ax.set_xticklabels(labels, fontsize=8)
ax.set_yticks(range(len(sets_))); ax.set_yticklabels(sets_, fontsize=8)
ax.set_xlim(-0.5, len(order)); ax.invert_yaxis()
ax.set_title("Median log2 change per set (blue up, red down; black ring = differs from random proteins, q < 0.05)", fontsize=9, loc="left")
plt.tight_layout(); plt.savefig(os.path.join(DATA_DIR, "chart_redox_sets.png"), dpi=150); plt.show()

In [ ]:
#@title Cell 4 - Follow-up tests: robustness, leave-one-out, size removal, reproducibility, unbiased GO screen
import os, re, numpy as np, pandas as pd
f = os.path.join(DATA_DIR, TABLE_FILE); out = DATA_DIR
d = pd.read_excel(f, sheet_name="Liver Proteomics")
d = d[d["Gene name"].notna()].copy()
d["gene"] = d["Gene name"].astype(str).str.split().str[0]
d = d.drop_duplicates("gene")
L2, L4, S, L = "large (2N) / small (2N)", "large (4N) / small (4N)", "4N (small) / 2N (small)", "4N (large) / 2N (large)"
d["ploidy_small"] = d[S]; d["ploidy_large"] = d[L]; d["dna_conc"] = d[S] - d[L2]; d["size_2n"] = d[L2]; d["size_4n"] = d[L4]
SETS = {
 "Glutathione S-transferases": "Gsta1 Gsta2 Gsta3 Gsta4 Gstm1 Gstm2 Gstm3 Gstm5 Gstm6 Gstm7 Gstp1 Gstp2 Gstk1 Gsto1 Gstt1 Gstt2 Gstt3 Gstz1 Mgst1 Mgst3",
 "Glutathione synthesis & recycling": "Gclc Gclm Gss Gsr Gpx1 Gpx4 Glrx Glrx2 Glrx3 Glrx5",
 "Thioredoxin / peroxiredoxin": "Txn1 Txnrd1 Txn2 Txnrd2 Prdx1 Prdx2 Prdx3 Prdx4 Prdx5 Prdx6 Srxn1 Txnl1",
 "H2O2 / ROS-producing enzymes": "Cyp2e1 Xdh Maoa Maob Ero1a Ero1lb Acox1 Acox2 Acox3 Hao1 Hao2 Dao Aox1 Nox4",
 "NADPH supply": "G6pdx Pgd Tkt Taldo1 Me1 Me2 Idh1 Idh2 Nnt Mthfd1 Aldh1l1",
 "NRF2 targets (other)": "Nqo1 Hmox1 Ephx1 Cbr1 Cbr3 Akr7a5 Akr1b8 Ftl1 Fth1 Blvrb Ugdh",
 "Superoxide dismutase & catalase": "Sod1 Sod2 Sod3 Cat",
}
rng = np.random.default_rng(7)
def comp_label(r):
    if r.Mitochondria: return "mito"
    if r.ER: return "ER"
    if r.Nucleus and not r.Cytoplasm: return "nucleus"
    return "cyto"
d["comp"] = d.apply(comp_label, axis=1)

def set_test(dd, genes, col, n=5000, by_comp=False, stat=np.median):
    dd = dd.reset_index(drop=True)
    pos = {g: i for i, g in enumerate(dd.gene)}; ii = np.array([pos[g] for g in genes if g in pos])
    if len(ii) < 3: return np.nan, np.nan, len(ii)
    key = pd.qcut(dd.UniqID.rank(method="first"), 5, labels=False).astype(str)
    if by_comp: key = key + "|" + dd.comp
    pools = {k: g.index.to_numpy() for k, g in dd.groupby(key)}
    ks = key.to_numpy()[ii]; v = dd[col].to_numpy()
    null = np.array([stat(v[[rng.choice(pools[k]) for k in ks]]) for _ in range(n)])
    obs = stat(v[ii]); c = np.median(null)
    return obs, (1 + np.sum(np.abs(null - c) >= abs(obs - c))) / (n + 1), len(ii)

lines = []
def say(s=""): print(s); lines.append(s)
base = d[d.UniqID >= 2]
cols = ["ploidy_small", "ploidy_large", "dna_conc"]
say("TEST 1 - Robustness: stricter protein filter (>=5 peptides), mean instead of median, compartment-matched random sets")
rows = []
for name, gs in SETS.items():
    gl = gs.split()
    for col in cols:
        r = {"set": name, "comparison": col}
        r["base_med"], r["base_p"], r["n"] = set_test(base, gl, col)
        r["pep5_med"], r["pep5_p"], r["n5"] = set_test(d[d.UniqID >= 5], gl, col)
        r["mean"], r["mean_p"], _ = set_test(base, gl, col, stat=np.mean)
        r["comp_med"], r["comp_p"], _ = set_test(base, gl, col, by_comp=True)
        rows.append(r)
t1 = pd.DataFrame(rows); pd.set_option("display.width", 260)
for col in cols:
    say(f"\n  {col}"); say(t1[t1.comparison == col].drop(columns="comparison").set_index("set").round(3).to_string())
t1.to_csv(os.path.join(out, "test1_robustness.csv"), index=False)

say("\nTEST 2 - Leave-one-out: is a set driven by one protein? (ploidy_small)")
for name in ["Glutathione S-transferases", "Glutathione synthesis & recycling", "Thioredoxin / peroxiredoxin", "H2O2 / ROS-producing enzymes"]:
    gl = [g for g in SETS[name].split() if g in set(base.gene)]
    meds = []
    for g in gl:
        m, p, _ = set_test(base, [x for x in gl if x != g], "ploidy_small", n=2000); meds.append((g, m, p))
    worst = max(meds, key=lambda x: x[2])
    say(f"  {name}: medians without each protein {min(m for _, m, _ in meds):+.2f} to {max(m for _, m, _ in meds):+.2f}; "
        f"largest p {worst[2]:.3f} (when {worst[0]} is left out)")
    frac = (base.set_index("gene").loc[gl, "ploidy_small"] < 0).mean()
    say(f"     {int(round(frac * len(gl)))} of {len(gl)} proteins lower in 4n (all proteins: {(base.ploidy_small < 0).mean():.0%})")

say("\nTEST 3 - Remove what cell size alone predicts. Small 4n cells are 2x bigger than small 2n, so take out each protein's")
say("size scaling (mean of the 2n and 4n size ratios) and test what remains.")
size = (base.size_2n + base.size_4n) / 2
b = np.polyfit(size, base.ploidy_small, 1)[0]
base = base.assign(ploidy_small_resid=base.ploidy_small - b * size)
say(f"  Across all proteins, ploidy_small = {b:+.3f} x size scaling + rest")
for name, gs in SETS.items():
    m, p, n = set_test(base, gs.split(), "ploidy_small_resid")
    say(f"  {name:36s} n={n:2d}  after removing size: {m:+.3f}  p={p:.4f}")

say("\nTEST 4 - Reproducibility: do small-cell and large-cell ploidy comparisons agree, protein by protein?")
say(f"  All proteins: r = {base.ploidy_small.corr(base.ploidy_large):.2f}; proteins with >=10 peptides: "
    f"r = {base[base.UniqID >= 10].ploidy_small.corr(base[base.UniqID >= 10].ploidy_large):.2f}")
allg = sorted({g for v in SETS.values() for g in v.split()} & set(base.gene))
x = base.set_index("gene").loc[allg]
say(f"  Redox enzymes ({len(allg)}): r = {x.ploidy_small.corr(x.ploidy_large):.2f}; lower in 4n in both: "
    f"{int(((x.ploidy_small < 0) & (x.ploidy_large < 0)).sum())}, higher in both: {int(((x.ploidy_small > 0) & (x.ploidy_large > 0)).sum())}")

say("\nTEST 5 - Unbiased screen: every GO biological process with >=10 proteins, ranked by ploidy_small (4n vs 2n, same DNA per volume)")
go = {}
for g, s in zip(base.gene, base["Gene ontology (biological process)"].fillna("")):
    for term in re.findall(r"([^;]+?) \[GO:\d+\]", s):
        go.setdefault(term.strip(), []).append(g)
go = {k: v for k, v in go.items() if 10 <= len(v) <= 300}
z = (base.ploidy_small - base.ploidy_small.mean()) / base.ploidy_small.std()
zs = pd.Series(z.to_numpy(), index=base.gene)
from scipy.stats import mannwhitneyu
rows = []
for term, gs in go.items():
    inn = zs.loc[gs]; outn = zs.drop(gs)
    p = mannwhitneyu(inn, outn).pvalue
    rows.append({"GO term": term, "n": len(gs), "median_log2": base.set_index("gene").loc[gs, "ploidy_small"].median(), "p": p})
g5 = pd.DataFrame(rows).sort_values("p")
p = g5.p.to_numpy(); o = np.argsort(p); q = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
g5["q"] = np.minimum(q, 1)[np.argsort(o)] if False else np.minimum(q, 1)
g5 = g5.reset_index(drop=True); g5["rank"] = np.arange(1, len(g5) + 1)
say(f"  {len(g5)} GO terms tested; {int((g5.q < 0.05).sum())} with q < 0.05")
say(g5.head(25)[["rank", "GO term", "n", "median_log2", "q"]].round(4).to_string(index=False))
red = g5[g5["GO term"].str.contains("glutathione|oxidative|oxidant|redox|hydrogen peroxide|reactive oxygen|xenobiotic|NADP", case=False)]
say("\n  Redox-related GO terms and their rank:")
say(red[["rank", "GO term", "n", "median_log2", "q"]].round(4).to_string(index=False))
g5.to_csv(os.path.join(out, "test5_go_screen_ploidy_small.csv"), index=False)
open(os.path.join(out, "follow_up_results.txt"), "w").write("\n".join(lines))
